In [1]:
import pennylane as qml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy as sp

In [2]:
df = pd.read_csv('C:\\Users\\jayam\\OneDrive\\Documents\\ml\\qml\\SVM\\Cancer_Data.csv')

In [3]:
x_data = df.to_numpy()

In [4]:
x_data

array([[842302, 'M', 17.99, ..., 0.4601, 0.1189, nan],
       [842517, 'M', 20.57, ..., 0.275, 0.08902, nan],
       [84300903, 'M', 19.69, ..., 0.3613, 0.08758, nan],
       ...,
       [926954, 'M', 16.6, ..., 0.2218, 0.0782, nan],
       [927241, 'M', 20.6, ..., 0.4087, 0.124, nan],
       [92751, 'B', 7.76, ..., 0.2871, 0.07039, nan]],
      shape=(569, 33), dtype=object)

In [5]:
x_data = x_data[:,2:32]
x_data = np.array(x_data,dtype = float)

In [6]:
x_data

array([[1.799e+01, 1.038e+01, 1.228e+02, ..., 2.654e-01, 4.601e-01,
        1.189e-01],
       [2.057e+01, 1.777e+01, 1.329e+02, ..., 1.860e-01, 2.750e-01,
        8.902e-02],
       [1.969e+01, 2.125e+01, 1.300e+02, ..., 2.430e-01, 3.613e-01,
        8.758e-02],
       ...,
       [1.660e+01, 2.808e+01, 1.083e+02, ..., 1.418e-01, 2.218e-01,
        7.820e-02],
       [2.060e+01, 2.933e+01, 1.401e+02, ..., 2.650e-01, 4.087e-01,
        1.240e-01],
       [7.760e+00, 2.454e+01, 4.792e+01, ..., 0.000e+00, 2.871e-01,
        7.039e-02]], shape=(569, 30))

In [7]:
y_data = np.array(df['diagnosis'])
y_data = np.where(y_data=='M',1,-1)

In [8]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x_data,y_data)
x_mean = np.sum(x_train,axis=0)/x_train.shape[0]
x_std = np.std(x_train,axis=0)
x_train = (x_train-x_mean)/x_std
x_test = (x_test-x_mean)/x_std
print(x_train)
print(x_train.shape)

[[-0.29038339 -0.20350954 -0.36657932 ... -0.98144312 -1.4484093
  -1.24140669]
 [ 0.37825798 -0.46742495  0.50512659 ...  1.66907128  0.56632524
   1.99469589]
 [-0.36629901  0.72593169 -0.40425869 ... -0.31900628 -0.11370711
  -0.17153686]
 ...
 [-1.08165768 -0.67855728 -1.09899856 ... -1.38522583 -0.34831034
  -0.54807693]
 [ 0.71111875  0.18433137  0.76337846 ...  1.37145029  0.24295322
   0.64693047]
 [-0.51229057 -0.31825537 -0.54439208 ... -0.28817029  0.87860117
  -0.83160934]]
(426, 30)


In [9]:
def making_kernel(X):
    X = X.copy()
    y = np.zeros((426,2))
    X=np.hstack((X,y))
    z = np.zeros((86,32))
    X=np.vstack((X,z))
    state_vector = X.flatten()
    norm = np.linalg.norm(state_vector)
    normalized_state_vector = state_vector/norm
    samples,features = X.shape
    index_bits = int(np.ceil(np.log2(samples)))
    data_bits = int(np.ceil(np.log2(features)))
    total_bits = index_bits+data_bits
    dev = qml.device('default.qubit',wires=total_bits)
    index_wires = list(range(index_bits))
    data_wires = list(range(index_bits,total_bits))
    def quantum_circuit(X):
        qml.StatePrep(normalized_state_vector,wires=index_wires+data_wires)
        return qml.density_matrix(wires=index_wires)
    circuit = qml.QNode(quantum_circuit,dev)
    partial_trace = circuit(X)
    Kernel_matrix = partial_trace*(norm**2)
    return np.real(Kernel_matrix[:426,:426])

In [10]:
kernel_matrix = making_kernel(x_train)
p = np.outer(y_train,y_train)*kernel_matrix
q=np.ones((x_train.shape[0],1))*-1
print(p)

[[ 25.6990267   27.50993396  -7.52459768 ...  13.21888582  11.70661902
    9.60940817]
 [ 27.50993396  52.06737419 -13.91573099 ...  19.16618929  17.93194047
   14.22295121]
 [ -7.52459768 -13.91573099   8.44566706 ...  -7.50050849  -5.02557014
   -4.49400684]
 ...
 [ 13.21888582  19.16618929  -7.50050849 ...  19.88684424  14.14908154
    7.80756219]
 [ 11.70661902  17.93194047  -5.02557014 ...  14.14908154  14.74923458
    6.7629408 ]
 [  9.60940817  14.22295121  -4.49400684 ...   7.80756219   6.7629408
    9.65803071]]


In [11]:
def find_alphas(p,q,alpha):
    C=1 #model parameter to be choosen by trainer
    def function(alpha):
        quad_part = 0.5*np.dot(alpha,np.dot(p,alpha))
        linear_part = -np.sum(alpha)
        return quad_part+linear_part
    equality_constraint = {
        'type': 'eq',
        'fun': lambda alpha: np.dot(y_train, alpha) - 0  
    }
    bounds = [(0,C) for _ in range(x_train.shape[0])]
    initial_guess = np.zeros(x_train.shape[0])
    result =sp.optimize.minimize(
    fun=function,
    x0=initial_guess,
    method='SLSQP',       
    bounds=bounds,         
    constraints=[equality_constraint] 
    )
    return result.x

In [12]:
alp=np.zeros(x_train.shape[0])
alphas=find_alphas(p,q,alp)
print(alphas)

[0.00000000e+00 1.48192839e-12 1.00000000e+00 0.00000000e+00
 9.79494542e-13 0.00000000e+00 1.07391499e-12 2.05853451e-12
 9.33899055e-13 0.00000000e+00 4.13014684e-13 2.19971299e-13
 7.04946906e-14 1.09061033e-12 2.44126515e-12 0.00000000e+00
 0.00000000e+00 4.71478136e-01 1.68222976e-12 6.78964357e-13
 1.69997681e-12 3.17243492e-13 0.00000000e+00 4.66402960e-12
 0.00000000e+00 0.00000000e+00 2.84709506e-13 0.00000000e+00
 0.00000000e+00 9.47132501e-13 0.00000000e+00 2.03810048e-13
 5.74464049e-13 9.57166584e-02 0.00000000e+00 1.45949192e-12
 1.29754766e-12 2.24544536e-13 4.62538303e-12 0.00000000e+00
 1.32206119e-12 1.30944010e-12 0.00000000e+00 0.00000000e+00
 2.70180812e-13 0.00000000e+00 1.10784102e-12 0.00000000e+00
 1.42683471e-12 2.13688872e-12 0.00000000e+00 0.00000000e+00
 1.11957087e-12 0.00000000e+00 0.00000000e+00 1.22124142e-12
 9.37755885e-13 0.00000000e+00 0.00000000e+00 0.00000000e+00
 5.15856347e-14 1.00000000e+00 1.00000000e+00 1.71136396e-12
 1.00000000e+00 1.016131

In [13]:
sv_indexes = alphas>1e-4
sv_indexes = np.where(sv_indexes)[0]
support_vectors = x_train[sv_indexes]
sv_labels = y_train[sv_indexes].flatten()
sv_alphas = alphas[sv_indexes].flatten()
print(f'support vectors indices are {sv_indexes}')

support vectors indices are [  2  17  33  61  62  64  71  73  87 107 119 127 128 131 147 149 158 180
 187 193 196 198 206 242 243 248 276 285 302 310 320 325 328 332 337 346
 410]


In [14]:
b_sum = 0
for k in range(len(support_vectors)):
    arr = [sv_alphas[i]*sv_labels[i]*kernel_matrix[sv_indexes[i]][sv_indexes[k]] for i in range(len(support_vectors))]
    pred = np.sum(np.array(arr))
    b_sum += (sv_labels[k]-pred)
bias = b_sum/len(sv_labels)
print(f'bias is {bias}')

bias is 0.022403471319664754


In [15]:
y_pred=[]
for i in range(x_test.shape[0]):
    test = x_test[i]
    arr = [sv_labels[k]*sv_alphas[k]*np.dot(test,support_vectors[k]) for k in range(len(sv_indexes))]
    pred = np.sum(np.array(arr))
    pred=pred-bias
    if pred>0:
      y_pred.append(1)
    else:
        y_pred.append(-1)
print(f'accuracy is {np.sum(y_pred==y_test)/len(y_test)}')

accuracy is 0.986013986013986
